In [2]:
%%time

# SuperFastPython.com
# example of thread-safe writing to a file with a dedicated writer thread
from random import random
from threading import Thread
from queue import Queue
 
# dedicated file writing task
def file_writer(filepath, queue):
    # open the file
    with open(filepath, 'w') as file:
        # run until the event is set
        while True:
            # get a line of text from the queue
            line = queue.get()
            # check if we are done
            if line is None:
                # exit the loop
                break
            # write it to file
            file.write(line)
            # flush the buffer
            file.flush()
            # mark the unit of work complete
            queue.task_done()
    # mark the exit signal as processed, after the file was closed
    queue.task_done()
 
# task for worker threads
def task(number, queue):
    # task loop
    for i in range(1000):
        # generate random number between 0 and 1
        value = random()
        # put the result in the queue
        queue.put(f'Thread {number} got {value}.\n')
 
# create the shared queue
queue = Queue()
# defile the shared file path
filepath = 'test_output.txt'
# create and start the file writer thread
writer_thread = Thread(target=file_writer, args=(filepath,queue), daemon=True)
writer_thread.start()
# configure worker threads
threads = [Thread(target=task, args=(i,queue)) for i in range(1000)]
# start threads
for thread in threads:
    thread.start()
# wait for threads to finish
for thread in threads:
    thread.join()
# signal the file writer thread that we are done
queue.put(None)
# wait for all tasks in the queue to be processed
queue.join()

CPU times: user 55.3 s, sys: 28.1 s, total: 1min 23s
Wall time: 1min 5s


In [3]:
%%time

# SuperFastPython.com
# example of thread-safe writing to a file
from random import random
from threading import Thread
from threading import Lock
 
# task for worker threads
def task(number, file, lock):
    # task loop
    for i in range(1000):
        # generate random number between 0 and 1
        value = random()
        # write to the file
        with lock:
            file.write(f'Thread {number} got {value}.\n')
 
# create the shared lock
lock = Lock()
# defile the shared file path
filepath = 'test_output2.txt'
# open the file
file = open(filepath, 'a')
# configure many threads
threads = [Thread(target=task, args=(i,file,lock)) for i in range(1000)]
# start threads
for thread in threads:
    thread.start()
# wait for threads to finish
for thread in threads:
    thread.join()
# close the file
file.close()

CPU times: user 16.5 s, sys: 11.8 s, total: 28.3 s
Wall time: 17.6 s
